# Author counts
How many lines does each author have in our two corpora? The plan wants about 20,000 lines per author.

In [1]:
import json
import re
import pandas as pd

DATA = "../data/"

## Load the data

In [2]:
gutenberg = pd.read_parquet(DATA + "gutenberg-poetry-corpus/data/train-00000-of-00001-fa9fb9e1f16eed7e.parquet")
catalog = pd.read_csv(DATA + "gutenberg-catalog/pg_catalog.csv", low_memory=False)

with open(DATA + "public-domain-poetry/poems.json", encoding="utf-8") as f:
    pdp = pd.DataFrame(json.load(f))

print(len(gutenberg), "Gutenberg lines")
print(len(pdp), "public-domain-poetry poems")

3085117 Gutenberg lines
38499 public-domain-poetry poems


## Find the author of each Gutenberg book
The corpus only has book IDs, so we look up the author in the Gutenberg catalogue.
Books with more than one author (anthologies) are skipped.

In [3]:
def author_key(first, last):
    # "Emily Elizabeth" + "Dickinson" -> "emily dickinson", so names match in both corpora
    if first:
        return (first.split()[0] + " " + last).lower()
    return last.lower()


def parse_authors(authors):
    # catalogue format: "Dickinson, Emily, 1830-1886; Todd, Mabel Loomis, 1856-1932 [Editor]"
    if not isinstance(authors, str):
        return None
    people = [a.strip().replace(" [Author]", "") for a in authors.split(";")]
    people = [a for a in people if "[" not in a]  # drop editors, translators, illustrators
    if len(people) != 1:
        return None
    names = [p for p in people[0].split(", ") if not re.search(r"\d", p)]
    if not names or names[0] in ["Various", "Anonymous", "Unknown"]:
        return None
    first = names[1] if len(names) > 1 else ""
    return author_key(first, names[0])


catalog["Text#"] = pd.to_numeric(catalog["Text#"], errors="coerce")
catalog = catalog.set_index("Text#")

books = pd.DataFrame({"book": gutenberg["gutenberg_id"].unique()})
books["author"] = books["book"].map(lambda b: parse_authors(catalog["Authors"].get(b)))
books = books.dropna()
print(len(books), "books with one author")

gutenberg = gutenberg.merge(books[["book", "author"]], left_on="gutenberg_id", right_on="book")

975 books with one author


## Authors in public-domain-poetry

In [4]:
pdp["text"] = pdp["text"].fillna("")
pdp = pdp[~pdp["Author"].str.strip().str.lower().isin(["various", "anonymous", "unknown"])]


def pdp_key(name):
    words = re.sub(r"\(.*?\)", "", name).split()  # drop things like "(Sir)"
    return author_key(" ".join(words[:-1]), words[-1])


pdp["author"] = pdp["Author"].map(pdp_key)

# one row per line
pdp_lines = pdp[["author"]].assign(line=pdp["text"].str.split("\n")).explode("line")

## Count lines per author
Some poems are in both corpora, and Gutenberg has some books twice (e.g. Dickinson's three series
*and* the complete edition). So we also count `unique_lines`: each distinct line once
(lowercase, letters only). That's the number that matters.

In [5]:
lines = pd.concat([gutenberg[["author", "line"]], pdp_lines])
lines["clean"] = lines["line"].fillna("").str.lower().str.replace(r"[^a-z ]", " ", regex=True)
lines["clean"] = lines["clean"].str.split().str.join(" ")
lines = lines[lines["clean"] != ""]

counts = lines.groupby("author").agg(lines=("clean", "size"), unique_lines=("clean", "nunique"))
counts["gutenberg_books"] = books.groupby("author").size()
counts["pdp_poems"] = pdp.groupby("author").size()
counts = counts.fillna({"gutenberg_books": 0, "pdp_poems": 0}).sort_values("unique_lines", ascending=False)
counts[["gutenberg_books", "pdp_poems"]] = counts[["gutenberg_books", "pdp_poems"]].astype(int)
counts.to_csv(DATA + "author_counts.csv")
print(len(counts), "authors")

854 authors


## Our candidate authors

In [6]:
candidates = ["emily dickinson", "christina rossetti", "edgar poe", "john keats", "walt whitman", "john milton"]
counts.loc[candidates]

,lines,unique_lines,gutenberg_books,pdp_poems
author,,,,
emily dickinson,16409,5537,4,448
christina rossetti,11846,11227,0,242
edgar poe,10493,5460,3,87
john keats,21666,12935,3,147
walt whitman,20587,13826,2,338
john milton,58402,28473,6,131


## Authors with the most lines
Possible replacements. Skip translations (Homer, Dante, Virgil, ...), since they're not the poet's own style.

In [7]:
counts.head(40)

,lines,unique_lines,gutenberg_books,pdp_poems
author,,,,
homer,123554,121691,11,0
robert browning,72709,58903,4,214
william wordsworth,75212,58781,3,969
burton stevenson,61844,58724,4,0
william cowper,60483,54215,1,308
valmiki,54234,53432,1,0
madison cawein,90806,51260,13,1356
dante alighieri,137189,49435,34,100
henry longfellow,98668,45674,5,296


## Findings (10 Nov)
- Only Milton has 20k+ unique lines. Dickinson 5.5k, Rossetti 11.2k, Poe 5.5k, Keats 12.9k, Whitman 13.8k.
- Rossetti has no books in the Gutenberg corpus; all her lines come from public-domain-poetry.
- Options with more data: Swinburne, Tennyson, Longfellow, Shelley, Wordsworth, Edgar Lee Masters.
- Matching on first + last name merges a few different poets who share a name (two James Thomsons;
  Thomas Moore and Thomas Sturge Moore). None of our candidates is affected.
- These are counts of the raw data, before `clean_pdp.ipynb`.
- To decide: lower the target to ~10k, or swap authors.